# **Model C — XGBoost on 12 Librosa Features (SI Train/Val/Test — 5-Block CV)**
SI Dataset 5-Block Rotating CV (Subject-Level StratifiedKFold) · XGBoost on 12 librosa features


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
import random
import warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import soundfile as sf
import librosa

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve, auc,
    average_precision_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Global Seed ───────────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Seed set to {RANDOM_SEED}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
BASE_DIR    = Path('..')

# SI data (train / val / test — subject-level 5-block CV)
NON_TB_PATH = BASE_DIR / 'SI_DATA' / 'Cough_sounds_healthy_individuals'
TB_PATH     = BASE_DIR / 'SI_DATA' / 'Cough_sounds_patients_with_ptb'

# Audio
SAMPLE_RATE   = 16000
CLIP_DURATION = 2
CLIP_LENGTH   = int(SAMPLE_RATE * CLIP_DURATION)

# Librosa feature extraction — paper-specified parameters
N_MFCC         = 12
N_MELS         = 40
N_FFT_PAPER    = 32768
N_FFT_FALLBACK = 2048

# Parallel feature extraction workers
N_WORKERS = 8

# CV — SI 5-block rotating subject-level StratifiedKFold
N_FOLDS     = 5
RANDOM_SEED = 42

# XGBoost hyperparameters
XGB_N_ESTIMATORS     = 200
XGB_MAX_DEPTH        = 3
XGB_LEARNING_RATE    = 0.1
XGB_SUBSAMPLE        = 0.8
XGB_COLSAMPLE_BYTREE = 1.0
XGB_EARLY_STOPPING   = 20

INFERENCE_BATCH_SIZE = 384

OUTPUT_DIR = BASE_DIR / 'output_model_c'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration loaded.')
print(f'Feature extraction workers : {N_WORKERS}')
print(f'Inference batch size       : {INFERENCE_BATCH_SIZE}')


## Load SI Dataset (Train / Val / Test — 5-Block Subject CV)


In [ ]:
# ── Load SI data from directory scan ─────────────────────────────────────────
def scan_audio(directory: Path) -> List[str]:
    paths = []
    with os.scandir(str(directory)) as entries:
        for e in entries:
            if e.is_file() and e.name.endswith(('.wav', '.mp3')):
                paths.append(e.path)
    return paths


tb_files     = scan_audio(TB_PATH)
non_tb_files = scan_audio(NON_TB_PATH)

rows = []
for f in tb_files:
    rows.append({'file_path': str(f), 'subject_id': Path(f).name.split('_')[0], 'label': 1})
for f in non_tb_files:
    rows.append({'file_path': str(f), 'subject_id': Path(f).name.split('_')[0], 'label': 0})

df_si = pd.DataFrame(rows)
df_si['subject_id'] = df_si['subject_id'].astype(str)
df_si['label']      = df_si['label'].astype('int8')

assert df_si.groupby('subject_id')['label'].nunique().max() == 1, \
    'SI: a subject has mixed labels!'

print(f'SI · Total clips   : {len(df_si):,}')
print(f'  Unique subjects  : {df_si["subject_id"].nunique():,}')
print(f'  TB-positive (1)  : {(df_si["label"]==1).sum():,}')
print(f'  TB-negative (0)  : {(df_si["label"]==0).sum():,}')
df_si.head(3)

## Feature Extraction (12 Librosa Features)

In [ ]:
# ── Audio loader ──────────────────────────────────────────────────────────────
def load_audio_array(path: str) -> np.ndarray:
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    if len(audio) < CLIP_LENGTH:
        audio = np.pad(audio, (0, CLIP_LENGTH - len(audio)))
    else:
        audio = audio[:CLIP_LENGTH]
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val
    return audio.astype(np.float32)


# ── Feature extraction — paper-specified parameters ───────────────────────────
# Target: 212-dim vector
# Base dims: MFCC(12)+RMS(1)+Centroid(1)+Bandwidth(1)+Rolloff(1)+ZCR(1)
#            +Mel(40)+ChromaSTFT(12)+ChromaCQT(12)+ChromaCENS(12)
#            +SpectralContrast(7)+Tonnetz(6) = 106
# Aggregation: mean(axis=1)[106] + std(axis=1)[106] = 212 total

def extract_features(audio: np.ndarray, sr: int) -> np.ndarray:
    # ── Paper-specified window / hop / n_fft ──────────────────────────────────
    win_length = int(sr * 0.050)   # 50 ms window
    hop_length = int(sr * 0.025)   # 25 ms hop  (50% overlap)
    window     = 'hamming'
    # Use paper's large FFT (zero-padded); fall back only if audio shorter than one window
    n_fft = N_FFT_PAPER if len(audio) >= win_length else N_FFT_FALLBACK

    stft_kw = dict(n_fft=n_fft, hop_length=hop_length,
                   win_length=win_length, window=window)

    feat_mats = []  # list of 2-D arrays (n_dims, T)

    # 1. MFCC — 12 coefficients, 40 Mel filters
    feat_mats.append(
        librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC, n_mels=N_MELS, **stft_kw)
    )

    # 2. RMS Energy — frame-based, no STFT
    feat_mats.append(
        librosa.feature.rms(y=audio, frame_length=win_length, hop_length=hop_length)
    )

    # 3. Spectral Centroid
    feat_mats.append(
        librosa.feature.spectral_centroid(y=audio, sr=sr, **stft_kw)
    )

    # 4. Spectral Bandwidth
    feat_mats.append(
        librosa.feature.spectral_bandwidth(y=audio, sr=sr, **stft_kw)
    )

    # 5. Spectral Rolloff — roll_percent=0.95 per paper
    feat_mats.append(
        librosa.feature.spectral_rolloff(y=audio, sr=sr, roll_percent=0.95, **stft_kw)
    )

    # 6. Zero Crossing Rate — frame-based, no STFT
    feat_mats.append(
        librosa.feature.zero_crossing_rate(y=audio, frame_length=win_length,
                                           hop_length=hop_length)
    )

    # 7. Mel Spectrogram — 40 filters, converted to dB
    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, **stft_kw)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feat_mats.append(mel_db)

    # 8. Chroma STFT — 12 bins (librosa default n_chroma=12)
    feat_mats.append(
        librosa.feature.chroma_stft(y=audio, sr=sr, **stft_kw)
    )

    # 9. Chroma CQT — 12 bins (does not use n_fft; librosa defaults)
    feat_mats.append(
        librosa.feature.chroma_cqt(y=audio, sr=sr, hop_length=hop_length)
    )

    # 10. Chroma CENS — 12 bins (does not use n_fft; librosa defaults)
    feat_mats.append(
        librosa.feature.chroma_cens(y=audio, sr=sr, hop_length=hop_length)
    )

    # 11. Spectral Contrast — 7 dims (librosa default: n_bands=6 → 7 output dims)
    feat_mats.append(
        librosa.feature.spectral_contrast(y=audio, sr=sr, **stft_kw)
    )

    # 12. Tonnetz — 6 dims (operates on harmonic component)
    harmonic = librosa.effects.harmonic(audio)
    feat_mats.append(
        librosa.feature.tonnetz(y=harmonic, sr=sr)
    )

    # ── Aggregate: mean and std across time axis for every feature matrix ─────
    means = np.concatenate([m.mean(axis=1) for m in feat_mats])   # (106,)
    stds  = np.concatenate([m.std(axis=1)  for m in feat_mats])   # (106,)
    vec   = np.concatenate([means, stds])                         # (212,)
    vec   = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
    return vec.astype(np.float32)


print('Feature extraction functions defined (paper-specified parameters).')

# Verify feature dimension on a silent dummy clip
_dummy = np.zeros(CLIP_LENGTH, dtype=np.float32)
FEATURE_DIM = len(extract_features(_dummy, SAMPLE_RATE))
print(f'Feature vector dimension: {FEATURE_DIM}  (expected: 212)')
assert FEATURE_DIM == 212, f'Dimension mismatch: got {FEATURE_DIM}, expected 212'


In [ ]:
# ── Parallel feature extraction helpers ───────────────────────────────────────
from tqdm.auto import tqdm
from joblib import Parallel, delayed


def _extract_one(path: str) -> np.ndarray:
    """Single-file worker: load audio → 212-dim feature vector."""
    audio = load_audio_array(path)
    return extract_features(audio, SAMPLE_RATE)


def parallel_extract(paths, desc: str, n_jobs: int = N_WORKERS) -> np.ndarray:
    """Extract features in parallel; returns float32 array (N, FEATURE_DIM)."""
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=0)(
        delayed(_extract_one)(p) for p in tqdm(paths, desc=desc)
    )
    return np.vstack(results).astype(np.float32)


# ── Extract SI features (with disk cache) ─────────────────────────────────────
CACHE_SI = Path(OUTPUT_DIR) / 'cache_features_si.npy'

if CACHE_SI.exists():
    features_si = np.load(str(CACHE_SI))
    print(f'SI features loaded from cache  : {features_si.shape}')
else:
    print(f'Extracting SI features ({len(df_si):,} clips, {N_WORKERS} workers) …')
    features_si = parallel_extract(df_si['file_path'].values, desc='SI Features')
    np.save(str(CACHE_SI), features_si)
    print(f'Saved → {CACHE_SI}  shape: {features_si.shape}')

labels_si = df_si['label'].values.astype(int)
print(f'SI feature matrix : {features_si.shape}')


## SI 5-Block Subject-Level CV Split


In [ ]:
# ── Build SI-only 5-block subject-level splits ────────────────────────────────

def _subject_labels(df, subj_col, label_col):
    return (
        df.groupby(subj_col)[label_col]
        .agg(lambda x: int(x.value_counts().idxmax()))
        .reset_index()
        .rename(columns={label_col: 'subject_label'})
        .sort_values(subj_col)
        .reset_index(drop=True)
    )


def build_si_folds(df_si, n_folds=N_FOLDS, seed=RANDOM_SEED):
    st_si = _subject_labels(df_si, 'subject_id', 'label')
    X_si  = st_si['subject_id'].values
    y_si  = st_si['subject_label'].values

    block_split = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    cv_folds = []

    for fold, (tr_idx, te_idx) in enumerate(block_split.split(X_si, y_si), start=1):
        tr_subj = X_si[tr_idx]
        te_subj = X_si[te_idx]

        inner_split = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed + fold)
        y_tr = y_si[tr_idx]
        val_inner_idx, train_inner_idx = next(
            (te, tr) for tr, te in inner_split.split(tr_subj, y_tr)
        )
        val_subj   = tr_subj[val_inner_idx]
        train_subj = tr_subj[train_inner_idx]

        assert set(train_subj).isdisjoint(val_subj),  f'Fold {fold}: train/val overlap!'
        assert set(train_subj).isdisjoint(te_subj),   f'Fold {fold}: train/test overlap!'
        assert set(val_subj).isdisjoint(te_subj),     f'Fold {fold}: val/test overlap!'

        cv_folds.append({
            'fold':       fold,
            'seed':       seed + fold,
            'train_subj': sorted(train_subj.tolist()),
            'val_subj':   sorted(val_subj.tolist()),
            'test_subj':  sorted(te_subj.tolist()),
        })

    return cv_folds


cv_folds = build_si_folds(df_si)

print('=' * 70)
print(f'Built {len(cv_folds)} SI-only folds (subject-level, no data leakage)')
print('  Train / Val / Test : all from SI subjects (disjoint per fold)')
print('=' * 70)

for fd in cv_folds:
    n_tr  = df_si[df_si['subject_id'].isin(fd['train_subj'])].shape[0]
    n_val = df_si[df_si['subject_id'].isin(fd['val_subj'])].shape[0]
    n_te  = df_si[df_si['subject_id'].isin(fd['test_subj'])].shape[0]
    print(f'Fold {fd["fold"]}  →  train {n_tr:,} clips | val {n_val:,} clips | test {n_te:,} clips')


In [ ]:
# ── Fold Summary (SI train / val / test) ─────────────────────────────────────

def _fmt_ids(ids: list, max_show: int = 6) -> str:
    if len(ids) <= max_show:
        return ', '.join(str(i) for i in ids)
    shown = ', '.join(str(i) for i in ids[:max_show])
    return f'{shown}  … (+{len(ids) - max_show} more)'


def print_si_fold_summary(cv_folds, df_si):
    n_si = df_si['subject_id'].nunique()
    W = 82

    print('=' * W)
    print(f'  FOLD SUMMARY  ·  {len(cv_folds)}-Block SI CV  ·  Subject-Level Stratified Split')
    print(f'  SI subjects : {n_si}  |  '
          f'TB+ {(df_si["label"]==1).sum():,} clips  |  TB- {(df_si["label"]==0).sum():,} clips')
    print('=' * W)

    for fd in cv_folds:
        fold = fd['fold']
        print(f'\n{"─"*W}')
        print(f'  FOLD {fold}')
        print(f'{"─"*W}')

        for split_name, key_subj in [('TRAIN', 'train_subj'), ('VAL  ', 'val_subj'), ('TEST ', 'test_subj')]:
            subj   = fd[key_subj]
            sub_df = df_si[df_si['subject_id'].isin(subj)]
            pct    = len(subj) / n_si * 100
            pos    = int((sub_df['label'] == 1).sum())
            neg    = int((sub_df['label'] == 0).sum())
            pos_ids = sorted(sub_df[sub_df['label'] == 1]['subject_id'].unique().tolist())
            neg_ids = sorted(sub_df[sub_df['label'] == 0]['subject_id'].unique().tolist())
            print(f'\n  {split_name} (SI)  {len(subj):>4} subjects ({pct:4.1f}%)  |'
                  f'  TB+ {pos:>5,} clips  |  TB- {neg:>5,} clips')
            print(f'    TB+ subjects [{len(pos_ids):>3}] : {_fmt_ids(pos_ids)}')
            print(f'    TB- subjects [{len(neg_ids):>3}] : {_fmt_ids(neg_ids)}')

    print(f'\n{"="*W}')
    print('  Train / Val / Test are subject-disjoint per fold')
    print('=' * W)


print_si_fold_summary(cv_folds, df_si)


## Training (XGBoost)

In [ ]:
# ── Helper: get (X, y) arrays from SI features for a set of subject IDs ───────
def get_si_fold_arrays(subject_ids):
    mask = df_si['subject_id'].isin(subject_ids).values
    return features_si[mask], labels_si[mask]


fold_results = []

print('\n' + '=' * 70)
print(f'Training Model C (XGBoost) — {N_FOLDS} Folds')
print('  Train/Val/Test: SI subject-level 5-block CV')
print('=' * 70)

for fold_info in cv_folds:
    fold = fold_info['fold']
    print(f'\n--- Fold {fold}/{N_FOLDS} ---')

    random.seed(RANDOM_SEED + fold)
    np.random.seed(RANDOM_SEED + fold)

    X_tr,  y_tr  = get_si_fold_arrays(fold_info['train_subj'])
    X_val, y_val = get_si_fold_arrays(fold_info['val_subj'])
    X_te,  y_te  = get_si_fold_arrays(fold_info['test_subj'])

    # StandardScaler: fit on SI train only
    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_te_s  = scaler.transform(X_te)

    # Class-imbalance weight
    n_neg = (y_tr == 0).sum()
    n_pos = (y_tr == 1).sum()
    scale_pos_weight = n_neg / n_pos
    print(f'  Healthy={n_neg}, PTB={n_pos}, scale_pos_weight={scale_pos_weight:.2f}')

    model = XGBClassifier(
        n_estimators          = XGB_N_ESTIMATORS,
        max_depth             = XGB_MAX_DEPTH,
        learning_rate         = XGB_LEARNING_RATE,
        subsample             = XGB_SUBSAMPLE,
        colsample_bytree      = XGB_COLSAMPLE_BYTREE,
        min_child_weight      = 1,
        gamma                 = 0.2,
        reg_alpha             = 0.01,
        reg_lambda            = 1.5,
        scale_pos_weight      = scale_pos_weight,
        random_state          = RANDOM_SEED + fold,
        n_jobs                = -1,
        eval_metric           = 'logloss',
        early_stopping_rounds = XGB_EARLY_STOPPING,
    )

    model.fit(
        X_tr_s, y_tr,
        eval_set = [(X_tr_s, y_tr), (X_val_s, y_val)],
        verbose  = False,
    )

    best_iter = model.best_iteration
    results   = model.evals_result()
    train_loss_curve = results['validation_0']['logloss']
    val_loss_curve   = results['validation_1']['logloss']

    print(f'  Early stopping at iteration {best_iter} '
          f'| val logloss={val_loss_curve[best_iter]:.4f}')

    fold_results.append({
        'fold': fold, 'model': model, 'scaler': scaler,
        'best_iter': best_iter,
        'train_loss': train_loss_curve,
        'val_loss':   val_loss_curve,
        'X_te_s': X_te_s, 'y_te': y_te,
        'test_subj': fold_info['test_subj'],
    })

print('\nTraining Complete.')


### Loss Curves

In [ ]:
fig, axes = plt.subplots(nrows=N_FOLDS, ncols=1, figsize=(8, 4 * N_FOLDS))
fig.suptitle('Training Curves — Model C (XGBoost, SI only)', fontsize=14, fontweight='bold', y=1.01)

for i, r in enumerate(fold_results):
    ax = axes[i]
    ep = range(1, len(r['train_loss']) + 1)
    ax.plot(ep, r['train_loss'], label='Train logloss', lw=2)
    ax.plot(ep, r['val_loss'],   label='Val logloss',   lw=2)
    ax.axvline(r['best_iter'] + 1, color='red', ls='--', lw=1.5,
               label=f'Best iter ({r["best_iter"]+1})')
    ax.set_title(f'Fold {r["fold"]}')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
# ── Collect predictions ───────────────────────────────────────────────────────
fold_probs   = []
fold_y_tests = []

for r in fold_results:
    probs = r['model'].predict_proba(r['X_te_s'])[:, 1]
    fold_probs.append(np.atleast_1d(probs))
    fold_y_tests.append(r['y_te'].astype(int))

print(f'Collected predictions from {len(fold_probs)} folds.')

In [ ]:

# ── Per-fold metrics ──────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print('SUMMARY — Model C (XGBoost librosa · SI 5-Block CV)')
print('=' * 70)

fold_summary = []
for i, (probs, y_true, r) in enumerate(zip(fold_probs, fold_y_tests, fold_results), start=1):
    fpr, tpr, thresholds = roc_curve(y_true, probs)
    j_idx  = np.argmax(tpr - fpr)
    thresh = float(thresholds[j_idx])

    preds          = (probs >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    sens  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    acc   = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    # Macro F1: average of F1 for each class
    f1_tb      = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0.0
    f1_healthy = 2*tn / (2*tn + fn + fp) if (2*tn + fn + fp) > 0 else 0.0
    f1_macro   = (f1_tb + f1_healthy) / 2
    auroc = roc_auc_score(y_true, probs)
    auprc = average_precision_score(y_true, probs)

    fold_summary.append({
        'Fold': i, 'AUROC': f'{auroc:.4f}', 'AUPRC': f'{auprc:.4f}',
        'Sensitivity': f'{sens:.3f}', 'Specificity': f'{spec:.3f}',
        'F1-Macro': f'{f1_macro:.3f}', 'Acc': f'{acc:.3f}',
        'Threshold': f'{thresh:.4f}', 'Best Iter': r['best_iter'] + 1,
        '_auroc': auroc, '_auprc': auprc, '_sens': sens, '_spec': spec,
        '_f1_macro': f1_macro, '_acc': acc,
    })

cols = ['Fold', 'AUROC', 'AUPRC', 'Sensitivity', 'Specificity', 'F1-Macro', 'Acc', 'Threshold', 'Best Iter']
print(pd.DataFrame(fold_summary)[cols].to_string(index=False))

aurocs    = [r['_auroc']    for r in fold_summary]
auprcs    = [r['_auprc']    for r in fold_summary]
senss     = [r['_sens']     for r in fold_summary]
specs     = [r['_spec']     for r in fold_summary]
f1_macros = [r['_f1_macro'] for r in fold_summary]
accs      = [r['_acc']      for r in fold_summary]

print(f'\nMean AUROC       : {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}')
print(f'Mean AUPRC       : {np.mean(auprcs):.4f} ± {np.std(auprcs):.4f}')
print(f'Mean Sensitivity : {np.mean(senss):.3f} ± {np.std(senss):.3f}')
print(f'Mean Specificity : {np.mean(specs):.3f} ± {np.std(specs):.3f}')
print(f'Mean F1 (Macro)  : {np.mean(f1_macros):.3f} ± {np.std(f1_macros):.3f}')
print(f'Mean Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}')

summary_json = {'model': 'model_c_xgboost_librosa_si_cv', 'folds': [], 'mean': {}}
for r in fold_summary:
    summary_json['folds'].append({
        'fold': r['Fold'], 'auroc': r['_auroc'], 'auprc': r['_auprc'],
        'sensitivity': r['_sens'], 'specificity': r['_spec'],
        'f1_macro': r['_f1_macro'], 'accuracy': r['_acc'],
        'best_iter': int(r['Best Iter']),
    })
summary_json['mean'] = {
    'auroc': float(np.mean(aurocs)), 'auprc': float(np.mean(auprcs)),
    'sensitivity': float(np.mean(senss)), 'specificity': float(np.mean(specs)),
    'f1_macro': float(np.mean(f1_macros)), 'accuracy': float(np.mean(accs)),
}
with open(f'{OUTPUT_DIR}/summary_model_c.json', 'w') as f:
    json.dump(summary_json, f, indent=2)
print(f'\nSaved to {OUTPUT_DIR}/summary_model_c.json')


### ROC Curves & Confusion Matrices

In [ ]:
FOLD_COLORS = ['#00BFFF', '#FFA500', '#FF0000', '#32CD32', '#8A2BE2']
mean_fpr = np.linspace(0, 1, 200)
fold_fprs, fold_tprs, fold_aucs = [], [], []

for y_true, probs in zip(fold_y_tests, fold_probs):
    fpr, tpr, _ = roc_curve(y_true, probs)
    fold_fprs.append(fpr); fold_tprs.append(tpr); fold_aucs.append(auc(fpr, tpr))

interp_tprs = [np.interp(mean_fpr, f, t) for f, t in zip(fold_fprs, fold_tprs)]
mean_tpr    = np.mean(interp_tprs, axis=0); mean_tpr[0] = 0.0
mean_auc    = auc(mean_fpr, mean_tpr); std_auc = np.std(fold_aucs)

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1], wspace=0.35)

ax_roc = fig.add_subplot(gs[0])
for i, (fpr, tpr, a) in enumerate(zip(fold_fprs, fold_tprs, fold_aucs)):
    ax_roc.plot(fpr, tpr, color=FOLD_COLORS[i], lw=1.2, ls='--',
                label=f'Fold {i+1} (AUC={a:.3f})')
ax_roc.plot(mean_fpr, mean_tpr, color='navy', lw=2,
            label=f'Mean (AUC={mean_auc:.3f}±{std_auc:.3f})')
ax_roc.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('ROC — Model C (XGBoost librosa, SI 5-Fold CV)'); ax_roc.legend(fontsize=9)
ax_roc.grid(alpha=0.3)

gs_cm = gridspec.GridSpecFromSubplotSpec(N_FOLDS, 1, subplot_spec=gs[1], hspace=0.55)
for i, (y_true, probs) in enumerate(zip(fold_y_tests, fold_probs)):
    fpr, tpr, ths = roc_curve(y_true, probs)
    thresh = float(ths[np.argmax(tpr - fpr)])
    preds = (probs >= thresh).astype(int)
    cm = confusion_matrix(y_true, preds)
    ax = fig.add_subplot(gs_cm[i])
    ax.imshow(cm, cmap=plt.cm.Blues, vmin=0)
    ax.set_title(f'Fold {i+1} (thr={thresh:.3f})', fontsize=8)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Healthy','TB'], fontsize=7)
    ax.set_yticks([0,1]); ax.set_yticklabels(['Healthy','TB'], fontsize=7, rotation=90, va='center')
    mid = cm.max() / 2
    for r_i in range(2):
        for c_i in range(2):
            ax.text(c_i, r_i, str(cm[r_i, c_i]), ha='center', va='center', fontsize=9,
                    color='white' if cm[r_i, c_i] > mid else 'black')

plt.suptitle('Model C (XGBoost librosa) Evaluation — SI 5-Fold CV', fontsize=13, y=1.01)
plt.savefig(f'{OUTPUT_DIR}/roc_cm_model_c.png', dpi=150, bbox_inches='tight')
plt.show()


### False Negative (FN) Error Analysis — Missed TB Segments

In [ ]:
# ── False Negative (FN) Error Analysis ─────────────────────────────────────────
print('\n' + '='*80)
print('FALSE NEGATIVE (FN) ERROR ANALYSIS — Missed TB Segments')
print('='*80)

for i, r in enumerate(fold_results):
    fold_num  = r['fold']
    model     = r['model']
    X_te_s    = r['X_te_s']
    y_true    = r['y_te'].astype(int)

    # Reconstruct per-fold test DataFrame
    df_test = df_si[df_si['subject_id'].isin(r['test_subj'])].reset_index(drop=True)

    probs = model.predict_proba(X_te_s)[:, 1]

    fpr_f, tpr_f, thresholds_f = roc_curve(y_true, probs)
    thresh = float(thresholds_f[np.argmax(tpr_f - fpr_f)])
    preds  = (probs >= thresh).astype(int)

    fn_mask = (y_true == 1) & (preds == 0)
    fn_df   = df_test[fn_mask][['subject_id', 'file_path', 'label']].copy()
    fn_df['prob'] = np.round(probs[fn_mask], 4)
    fn_df['pred'] = 0

    total_tb = (y_true == 1).sum()
    print(f'\n--- Fold {fold_num} | Threshold: {thresh:.4f} | FN: {fn_mask.sum()} / {total_tb} TB segments ---')
    if fn_mask.sum() > 0:
        print(fn_df.to_string(index=False))
    else:
        print('  No false negatives in this fold.')

print(f'\n{"="*80}')
